# Resolved Stellar Population UV–Optical
Run cells individually: download both surveys, inspect previews, exclude, stack, crop, PSF match, then mask. This notebook has not been executed. Coordinates below are user supplied. Distances are Mpc; sizes are arcseconds.


In [ ]:
# Install once, then restart the kernel.
# %pip install --upgrade git+https://github.com/akhilkrishnar0/resolved-stellar-population-uv-optical.git
import pandas as pd
from resolved_uv_optical import ResolvedPopulationPipeline
catalog=pd.DataFrame([("UGC6754",176.697,20.675,100.33,500),("UGC9024",211.669,22.070,33,1000),("NGC6902",306.117,-43.653,39.9,1000)],columns=["galaxy","ra_deg","dec_deg","distance","size"])
pipe=ResolvedPopulationPipeline()


In [ ]:
pipe.review_galex(catalog)
pipe.review_decals(catalog,size=3000)


Stop and inspect galex_review/ and decals_review/. Use download=False above for existing downloads. A 3000-pixel DECaLS cutout spans about 786 arcsec, so the requested 1000-arcsec fields have uncovered optical edges.


In [ ]:
exclude={"UGC6754":["AIS_226_0001_sg39-fd-int.fits","AIS_226_sg39-fd-int.fits","AIS_226_0001_sg39-nd-int.fits","AIS_226_0002_sg39-nd-int.fits","AIS_226_0003_sg39-nd-int.fits","AIS_226_sg39-nd-int.fits"]}
reject_decals=[]
approved=catalog.loc[~catalog.galaxy.isin(reject_decals)].copy()


The exclusion list is the user-supplied selection. Check it against your downloaded filenames; unknown filenames deliberately raise an error.


In [ ]:
pipe.run(approved,process_galex=True,galex_exclude=exclude,galex_previews=False,make_cutouts=False)


In [ ]:
pipe.run(approved,split_decals=True,make_cutouts=True)


In [ ]:
pipe.run(approved,make_cutouts=False,match_psf=True,do_plot=True,target_fwhm=5.3)


## Source masking and residual UV background
The runnable example below uses Gaia-matched detections without a protected ellipse: no measured galaxy geometries have been supplied. DAO-only fallback is disabled. Inspect central masks. To protect each galaxy, set protect_galaxy=True and supply its galaxy_sma_arcsec, galaxy_smb_arcsec and galaxy_pa_deg in per_galaxy. PA is east toward north (90 minus the usual north-through-east PA). Manual apertures use zero-based g-reference pixels.


In [ ]:
per_galaxy={g:{"manual_circles_refpix":[],"manual_ellipses_refpix":[]} for g in approved.galaxy}
# Add measured protection geometry and/or manual apertures to each entry above.
mask_report=pipe.mask_sources(approved,per_galaxy=per_galaxy,protect_galaxy=False,
    allow_dao_fallback=False,do_bgsub=True,make_plots=True)


Inspect masked/<galaxy>/ diagnostic PNGs and masking_report.json. UV background is a scalar clipped estimate from outer image corners, which must be free of galaxy light. Optical background-subtracted-named outputs are unchanged masked copies. Science grids remain different; no common-grid reprojection or uncertainty propagation is performed. Rerun masking after adjusting settings.


In [ ]:
from pathlib import Path
from IPython.display import Image,display
for galaxy in approved.galaxy:
    for png in sorted((pipe.base_dir/"masked"/galaxy).glob("*_mask_check.png")):
        display(Image(filename=str(png),width=650))
